# Train Per-Type Mask R-CNN Models (Fiber / Film / Fragment)

This notebook trains 3 specialised binary Mask R-CNN models on Google Colab
using SAM-annotated crops mounted from Google Drive.

## Google Drive Folder Structure

Upload your data to this structure **before running**:

```
MyDrive/
  mp-detect/
    data/
      crops_fiber_sam/
        images/          <- fiber crop images
        masks/           <- fiber SAM masks
        annotations.json
      crops_film_sam/
        images/          <- film crop images
        masks/           <- film SAM masks
        annotations.json
      crops_fragment_sam/
        images/          <- fragment crop images
        masks/           <- fragment SAM masks
        annotations.json
    models/              <- trained weights saved here automatically
      maskrcnn_fiber/
      maskrcnn_film/
      maskrcnn_fragment/
```

## 1. Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Base paths
DRIVE_ROOT = '/content/drive/MyDrive/mp-detect'
DATA_ROOT  = f'{DRIVE_ROOT}/data'
MODEL_ROOT = f'{DRIVE_ROOT}/models'

In [ ]:
!pip install -q albumentations opencv-python-headless

import os, json, random, cv2, numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Verify Data

In [ ]:
TYPES = ['fiber', 'film', 'fragment']

for t in TYPES:
    imgs_dir = Path(DATA_ROOT) / f'crops_{t}_sam' / 'images'
    masks_dir = Path(DATA_ROOT) / f'crops_{t}_sam' / 'masks'
    ann_file = Path(DATA_ROOT) / f'crops_{t}_sam' / 'annotations.json'
    n_imgs  = len(list(imgs_dir.glob('*.png')))  if imgs_dir.exists()  else 0
    n_masks = len(list(masks_dir.glob('*.png'))) if masks_dir.exists() else 0
    has_ann = ann_file.exists()
    status = '✅' if (n_imgs > 0 and n_masks > 0 and has_ann) else '❌'
    print(f'{status} {t:>10}: {n_imgs} images, {n_masks} masks, annotations={has_ann}')

print('\nEach model will train on ALL its available samples.')

## 3. Dataset & Model Definitions

In [ ]:
# ── Configuration ──
NUM_CLASSES = 2        # background + 1 target type
CROP_SIZE   = 128
BATCH_SIZE  = 8
LR          = 0.001
EPOCHS      = 50


# ── Dataset ──
class SingleTypeCropDataset(Dataset):
    def __init__(self, crops_dir, transforms=None, max_samples=None):
        self.crops_dir = Path(crops_dir)
        self.transforms = transforms
        self.images_dir = self.crops_dir / 'images'
        self.masks_dir  = self.crops_dir / 'masks'

        with open(self.crops_dir / 'annotations.json') as f:
            self.annotations = json.load(f)

        self.samples = [n for n in self.annotations if (self.images_dir / n).exists()]

        # Balanced sub-sampling
        if max_samples and max_samples < len(self.samples):
            random.seed(42)
            self.samples = sorted(random.sample(self.samples, max_samples))

        print(f'  [{self.crops_dir.name}] {len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        ann  = self.annotations[name]

        image = cv2.cvtColor(cv2.imread(str(self.images_dir / name)), cv2.COLOR_BGR2RGB)
        h, w  = image.shape[:2]

        # Load mask (SAM → default naming → ellipse fallback)
        mask = None
        mf = ann.get('mask_file')
        if mf and (self.masks_dir / mf).exists():
            raw = cv2.imread(str(self.masks_dir / mf), cv2.IMREAD_GRAYSCALE)
            if raw is not None: mask = (raw > 127).astype(np.uint8)
        if mask is None:
            dm = self.masks_dir / name.replace('.png', '_mask.png')
            if dm.exists():
                raw = cv2.imread(str(dm), cv2.IMREAD_GRAYSCALE)
                if raw is not None: mask = (raw > 127).astype(np.uint8)
        if mask is None:
            mask = np.zeros((h, w), np.uint8)
            rb = ann.get('rel_box')
            if rb:
                cx, cy = (rb[0]+rb[2])//2, (rb[1]+rb[3])//2
                ax, ay = (rb[2]-rb[0])//2, (rb[3]-rb[1])//2
            else:
                cx, cy = w//2, h//2
                ax, ay = int(w*0.4), int(h*0.4)
            if ax > 0 and ay > 0:
                cv2.ellipse(mask, (cx,cy), (ax,ay), 0, 0, 360, 1, -1)

        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        ys, xs = np.where(mask > 0)
        if len(xs) > 0:
            box = [xs.min(), ys.min(), xs.max(), ys.max()]
        else:
            m = min(h, w) // 10
            box = [m, m, w - m, h - m]

        boxes  = np.array([box], dtype=np.float32)
        labels = np.array([1], dtype=np.int64)
        masks  = np.array([mask], dtype=np.uint8)

        if self.transforms:
            t = self.transforms(image=image, bboxes=boxes.tolist(),
                                masks=list(masks), class_labels=labels.tolist())
            image = t['image']
            if len(t['bboxes']) > 0:
                boxes  = np.array(t['bboxes'], np.float32)
                labels = np.array(t['class_labels'], np.int64)
                masks  = np.array(t['masks'], np.uint8)
        else:
            image = torch.from_numpy(image.transpose(2,0,1)).float() / 255.0

        return image, {
            'boxes':   torch.as_tensor(boxes, dtype=torch.float32),
            'labels':  torch.as_tensor(labels, dtype=torch.int64),
            'masks':   torch.as_tensor(masks, dtype=torch.uint8),
            'image_id': torch.tensor([idx]),
            'area':    torch.as_tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes], dtype=torch.float32),
            'iscrowd': torch.zeros(len(boxes), dtype=torch.int64),
        }


# ── Transforms ──
def get_transforms(train=True):
    if train:
        return A.Compose([
            A.Resize(CROP_SIZE, CROP_SIZE),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.GaussNoise(std_range=(0.03, 0.15), p=0.3),
            A.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
            ToTensorV2(),
        ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))
    return A.Compose([
        A.Resize(CROP_SIZE, CROP_SIZE),
        A.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))


def collate_fn(batch):
    return tuple(zip(*batch))


# ── Model ──
def get_model(num_classes=NUM_CLASSES):
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    inf = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(inf, num_classes)
    inf_m = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(inf_m, 256, num_classes)
    return model

print('Definitions loaded ✅')

## 4. Training Function

In [ ]:
def train_type(mp_type):
    """Train a binary Mask R-CNN for one microplastic type using ALL its data."""
    crops_dir = f'{DATA_ROOT}/crops_{mp_type}_sam'
    save_dir  = f'{MODEL_ROOT}/maskrcnn_{mp_type}'
    os.makedirs(save_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'TRAINING — {mp_type.upper()}  (all available samples)')
    print(f'  crops : {crops_dir}')
    print(f'  save  : {save_dir}')
    print(f'{"="*60}')

    dataset = SingleTypeCropDataset(crops_dir, get_transforms(True))
    loader  = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=2,
                         collate_fn=collate_fn, pin_memory=True)

    model = get_model().to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optim  = torch.optim.AdamW(params, lr=LR, weight_decay=5e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS, eta_min=LR*0.01)

    best_loss = float('inf')
    history   = []

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0

        pbar = tqdm(loader, desc=f'[{mp_type}] Epoch {epoch+1}/{EPOCHS}', leave=False)
        for images, targets in pbar:
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            if not all(len(t['boxes']) > 0 for t in targets):
                continue

            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())

            optim.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optim.step()

            epoch_loss += losses.item()
            pbar.set_postfix(loss=f'{losses.item():.4f}')

        sched.step()
        avg = epoch_loss / max(len(loader), 1)
        history.append(avg)
        tag = ''

        ckpt = dict(epoch=epoch+1, mp_type=mp_type, num_classes=NUM_CLASSES,
                    model_state_dict=model.state_dict(),
                    optimizer_state_dict=optim.state_dict(), loss=avg)
        torch.save(ckpt, f'{save_dir}/maskrcnn_latest.pth')

        if avg < best_loss:
            best_loss = avg
            torch.save(ckpt, f'{save_dir}/maskrcnn_best.pth')
            tag = ' ⭐ best'

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:>3}/{EPOCHS}  loss={avg:.4f}{tag}')

    print(f'\n✅ {mp_type} done — best loss {best_loss:.4f}')
    print(f'   Saved to: {save_dir}/maskrcnn_best.pth')
    return history

print('Training function ready ✅')

ude there "## 5. Train All 3 Models

In [ ]:
# Show sample counts per type
for t in TYPES:
    d = Path(DATA_ROOT) / f'crops_{t}_sam' / 'images'
    n = len(list(d.glob('*.png'))) if d.exists() else 0
    print(f'  {t}: {n} samples')

print(f'\nTraining each model on ALL its available samples.\n')

# Train sequentially
all_history = {}
for mp_type in TYPES:
    all_history[mp_type] = train_type(mp_type)

print(f'\n{"="*60}')
print('ALL 3 MODELS TRAINED ✅')
for t in TYPES:
    print(f'  {t:>10}: {MODEL_ROOT}/maskrcnn_{t}/maskrcnn_best.pth')
print(f'{"="*60}')

## 6. Training Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for t in TYPES:
    ax.plot(range(1, len(all_history[t])+1), all_history[t], label=t)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Per-Type Mask R-CNN Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODEL_ROOT}/training_loss.png', dpi=150)
plt.show()
print(f'Saved: {MODEL_ROOT}/training_loss.png')

## 7. Quick Sanity Check — Predict on Random Samples

In [ ]:
import torchvision.transforms.functional as F

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row, mp_type in enumerate(TYPES):
    ckpt_path = f'{MODEL_ROOT}/maskrcnn_{mp_type}/maskrcnn_best.pth'
    ckpt = torch.load(ckpt_path, map_location=device)
    model = get_model().to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    imgs_dir = Path(DATA_ROOT) / f'crops_{mp_type}_sam' / 'images'
    imgs = sorted(imgs_dir.glob('*.png'))
    samples = random.sample(imgs, min(4, len(imgs)))

    for col, img_path in enumerate(samples):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        resized = cv2.resize(img, (CROP_SIZE, CROP_SIZE))
        tensor = F.to_tensor(resized).unsqueeze(0).to(device)

        with torch.no_grad():
            out = model(tensor)[0]

        ax = axes[row][col]
        ax.imshow(resized)

        if len(out['masks']) > 0:
            best = out['scores'].argmax()
            mask = out['masks'][best, 0].cpu().numpy() > 0.5
            score = out['scores'][best].item()
            ax.contour(mask, colors='lime', linewidths=1)
            ax.set_title(f'{mp_type} ({score:.2f})', fontsize=10)
        else:
            ax.set_title(f'{mp_type} (no det)', fontsize=10)
        ax.axis('off')

plt.suptitle('Per-Type Mask R-CNN Predictions', fontsize=14)
plt.tight_layout()
plt.savefig(f'{MODEL_ROOT}/sample_predictions.png', dpi=150)
plt.show()

## 8. Summary

| Model | Path |
|-------|------|
| Fiber | `MyDrive/mp-detect/models/maskrcnn_fiber/maskrcnn_best.pth` |
| Film | `MyDrive/mp-detect/models/maskrcnn_film/maskrcnn_best.pth` |
| Fragment | `MyDrive/mp-detect/models/maskrcnn_fragment/maskrcnn_best.pth` |

Download the `models/` folder or use these weights directly from Drive in your inference pipeline.